# Quantum Machine Learning for Air Pollution Forecasting

This Jupyter Notebook contains the reproducible experimental pipeline used in our research to evaluate Quantum Neural Networks (QNNs) for environmental time-series forecasting. 

It leverages **TensorFlow/Keras** for the classical neural network components and optimization, and **PennyLane** for building and simulating the parameterized quantum circuits (Ansatz).


In [ ]:
import os
import numpy as np
import tensorflow as tf
import pennylane as qml
import pandas as pd
from matplotlib import pyplot as plt

from utils.statistics import quantitative_analysis, get_mean_left_right_error_interval

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import LeakyReLU

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

tf.keras.backend.set_floatx('float32')
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)

### Environment and GPU Setup
To ensure stable training, especially when combining classical deep learning frameworks with quantum simulators, we configure TensorFlow to dynamically allocate GPU memory rather than hoarding the entirety of the VRAM.


In [ ]:
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
  # Restrict access of TensorFlow to specific GPU
  try:
    tf.config.experimental.set_visible_devices(gpus[0], 'GPU')
  except RuntimeError as e:
    # Visible devices must be set at program startup
    print(e)

In [ ]:
tf.config.experimental.get_visible_devices()

## Experimental Parameters and Plotting Functions

Here we define the core experimental parameters:
- `prevision_window`: The forecasting horizons (1 to 6 hours ahead).
- `lookback`: The number of past time steps used as input (1 or 2).
- `n_layers`: The depth of the quantum ansatz (repetitions of the variational-entanglement block).

We also define utility functions to visualize the training convergence (`plot_history`) and the final predictions with uncertainty bounds (`plot_prediction_versus_observed`).


In [ ]:
prevision_window = [1,2,3,4,5,6]
lookback = 1
batch_size = 128
#ansatz = "Wavelets"
ansatz = "QNN"
n_layers = 2

In [ ]:
SUBDIR = f"Local-Ansatz-{ansatz}-Lookback-{lookback}-Depth-{n_layers}"

In [ ]:
def plot_history(n_layers, history, ansatz, lookback, batch_size):
    plt.figure(figsize=(14,5), dpi=320, facecolor='w', edgecolor='k')
    plt.title(f"Quantum Model - {ansatz} - Loss for depth {n_layers} - Lookback {lookback}")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.plot(history.history['loss'], label="Loss/Epoch")
    plt.plot(history.history['val_loss'], label="Val Loss/Epoch")
    plt.xticks(range(0, len(history.history['loss'])+1, 30))
    plt.legend()
    plt.grid()

    path = os.path.abspath(os.path.join(os.getcwd(), 'plots', SUBDIR))
    os.makedirs(path, exist_ok=True)
    filename = f"loss-history-pollution-{ansatz}-{n_layers}-layers-lookback-{lookback}-batch-{batch_size}.png"
    print(f"Saving history in {os.path.join(path,filename)}")
    plt.savefig(os.path.join(path,filename))

    plt.show()

    values = np.array([list(range(1, len(history.history['loss'])+1)), history.history['loss'], history.history['val_loss']])
    loss_pd = pd.DataFrame(np.transpose(values))
    loss_pd.columns = ["Epoch", "Loss", "Val Loss"]
    loss_pd = loss_pd.set_index("Epoch")
    path = os.path.abspath(os.path.join(os.getcwd(), 'analysis', SUBDIR))
    os.makedirs(path, exist_ok=True)
    filename = f"loss-pollution-{ansatz}-{n_layers}-layers-lookback-{lookback}-batch-{batch_size}.csv"
    loss_pd.to_csv(os.path.join(path,filename))

In [ ]:
def plot_prediction_versus_observed(n_layers, y_test, y_pred, mean_error_normal, ansatz, lookback, batch_size, prev):

    for i in range(len(prev)):
        fig, ax = plt.subplots(figsize=(20,5), dpi=320, facecolor='w', edgecolor='k')
        plt.title(f"Quantum Model - {ansatz} - Lookback {lookback} - PM25 quantity for {prev[i]} hours ahead")
        plt.xlabel("Time")
        plt.ylabel("PM 2.5 (g/m\u00b3)")

        plt.plot(y_pred[:,i], label="Prediction", color='blue')
        plt.fill_between(range(y_pred.shape[0]), y_pred[:,i]-mean_error_normal[0,i], y_pred[:,i]+mean_error_normal[0,i], color='blue', alpha=0.05)
        plt.plot(y_test[:,i], label="Original", color='orange')
        plt.xticks(rotation=45)

        plt.legend()

        path = os.path.abspath(os.path.join(os.getcwd(), 'plots', SUBDIR))
        filename = f"prediction-pollution-{ansatz}-{n_layers}-layers-lookback-{lookback}-batch-{batch_size}-{prev[i]}-hours.png"
        print(f"Saving Prediction Plot in {os.path.join(path,filename)}")
        os.makedirs(path, exist_ok=True)
        plt.savefig(os.path.join(path,filename))

        plt.show()

## Data Loading and Formatting

We load the environmental datasets from the iSCAPE project (Sutherland and Stoke Park). The `load_table` function processes the raw CSV files, drops null values, and structures the temporal data according to the specified `lookback` window. If `lookback > 1`, older temporal identifiers are dropped to prevent target leakage.


In [ ]:
def load_table(path, prev, lookback):
    X = pd.read_csv(path)
    # Remove any rows with missing sensor readings to maintain temporal continuity
    X.dropna(axis=0,how='any',inplace=True)

    # We remove all outliers from dataset
    X = X[X['EXT_PM_25'] < 1000]

    if lookback > 1:
        for col in X.columns:
            X[col+'_Lookback'] = X.loc[:,col].shift(lookback-1)
        X = X.iloc[lookback-1:, :]

    # We copy the values in X to prepare the y dataset. The first row is removed from y
    # since it does not have a previous value to serve as forecast
    y = X[:].drop(X.index[0])

    # We remove the last line in X since it doesn't have an equivalent y
    X = X.iloc[:-prev[-1],:]

    # We create the final y dataset by creating a new column with the predictions and
    # removing the unnecessary information
    for i in prev:
        y[f'Prev {i} hour'] = y.loc[:,"EXT_PM_25"].shift(-(i-1))

    if prev[-1] == 1:
        y= y.iloc[:, -1:]
    else:
        y= y.iloc[:-(prev[-1]-1), -len(prev):]

    return X, y.values

In [ ]:
print("\nLoading Datasets\n")
path_stoke = os.path.abspath(os.path.join(os.getcwd(), 'data', 'stoke'))
path_suther = os.path.abspath(os.path.join(os.getcwd(), 'data', 'suther'))

filename_train = "wavelets-lvl5-train.csv" if ansatz == "Wavelets" else "train.csv"
filename_test  = "wavelets-lvl5-test.csv"  if ansatz == "Wavelets" else "test.csv"

train_file_stoke  = os.path.join(path_stoke, filename_train)
train_file_suther = os.path.join(path_suther, filename_train)
test_file_stoke   = os.path.join(path_stoke, filename_test)
test_file_suther  = os.path.join(path_suther, filename_test)

print(f"importing data from {train_file_stoke}")
X_train_stoke, y_train_stoke = load_table(train_file_stoke, prevision_window, lookback)
print(f"importing data from {train_file_suther}")
X_train_suther, y_train_suther = load_table(train_file_suther, prevision_window, lookback)

X_all = pd.concat([X_train_stoke, X_train_suther], axis=0)
y_all = np.vstack((y_train_stoke,y_train_suther))

print(f"importing data from {test_file_stoke}")
X_test_stoke, y_test_stoke = load_table(test_file_stoke, prevision_window, lookback)
print(f"importing data from {test_file_suther}")
X_test_suther, y_test_suther = load_table(test_file_suther, prevision_window, lookback)

X_test = pd.concat([X_test_stoke, X_test_suther], axis=0)
y_test = np.vstack((y_test_stoke,y_test_suther))


plot_time = X_test['Time'].values

In [ ]:
if lookback > 1:
# Drop redundant temporal string identifiers from lookback steps
    X_all = X_all.drop(["Time", "Month", "Time_Lookback", "Month_Lookback"], axis=1)
    X_test = X_test.drop(["Time", "Month", "Time_Lookback", "Month_Lookback"], axis=1)
else:
    X_all = X_all.drop(["Time", "Month"], axis=1)
    X_test = X_test.drop(["Time", "Month"], axis=1)

In [ ]:
print(f"\nThere are {X_all.shape[1]} features and {X_all.shape[0]} instances in All Train set\n")
print(X_all.head())
print(f"\nThere are {X_test.shape[1]} features and {X_test.shape[0]} instances in All Test set\n")
print(X_test.head())

In [ ]:
print("Size y_all", len(y_all),"\n", y_all[:5])
print("Size y_test_all", len(y_test),"\n", y_test[:5])
print("\n#########\n")

In [ ]:
# load dataset
values = X_all.values
# specify columns to plot
groups = [i for i in range(len(X_all.columns))]
i = 1
# plot each column
plt.figure(figsize=(20, 20))
for group in groups:
    plt.subplot(len(groups), 1, i)
    plt.plot(values[:, group])
    plt.title(X_all.columns[group], y=0.5, loc='right')
    i += 1
plt.show()

In [ ]:
X_all.max()

## Data Pre-processing

### Feature Normalization
Quantum feature maps typically encode classical data as rotations (angles) on qubits. Therefore, we scale all input features to a normalized range of $[0, 1]$ using `MinMaxScaler`. This ensures that all atmospheric variables and pollutant concentrations are weighted equally by the quantum embedding layer.


In [ ]:
print("\nScaling Data\n")
scaler_x = MinMaxScaler(feature_range=(0, 1))
Xs_all  = scaler_x.fit_transform(X_all)
Xs_test = scaler_x.transform(X_test)

In [ ]:
print(Xs_all[0:5])

In [ ]:
print(Xs_test[0:5])

### Spliting Train and Validation sets

In [ ]:
print("\nSplitting Train Data\n")
train_ratio = 0.7
Xs_train, Xs_val, y_train, y_val = train_test_split(Xs_all, y_all, test_size=1 - train_ratio, shuffle=False)

In [ ]:
print(f"There are {Xs_train.shape[1]} features and {Xs_train.shape[0]} instances in Train set")
print(f"There are {Xs_val.shape[1]} features and {Xs_val.shape[0]} instances in Val set")
print(f"There are {Xs_test.shape[1]} features and {Xs_test.shape[0]} instances in Test set")


## Quantum Circuit Definition (Ansatz)

This section defines the core Quantum Neural Network architecture using PennyLane. 

1. **Embedding:** The scaled classical data is loaded into the quantum state using `AngleEmbedding`, which applies Pauli-Y rotations based on the input features.
2. **Variational Layers:** We utilize `StronglyEntanglingLayers`, which consist of single-qubit rotations followed by a cascade of CNOT gates to build entanglement. The `n_layers` parameter dictates the depth of this circuit.


In [ ]:
n_qubits = Xs_train.shape[1] # n_features
print(f"\nCircuit size: {n_qubits} qubits\n")

In [ ]:
weight_shapes = {"weights": (n_layers, n_qubits, 3)}
weight_shapes

In [ ]:
def strong_entangling(inputs, weights):
    # weights: (n_layers,n_qubits,3)
    n_layers = len(weights)
    n_qubits = len(weights[0])

    ###############
    # Feature Map #
    ###############
    for idx in range(n_qubits):
        qml.Hadamard(wires=idx)
    qml.templates.AngleEmbedding(inputs, rotation='Y', wires=range(n_qubits))

    ##########
    # Ansatz #
    ##########
    for k in range(n_layers):
        # Variational Layer
        for i in range(len(weights[k])):
            qml.Rot(*weights[k][i],wires=i)

        # Entangling Layer
        for i in range(0, n_qubits-1):
            qml.CNOT(wires=[i, i + 1])
        qml.CNOT(wires=[n_qubits-1, 0])

    ###############
    # Measurement #
    ###############
    return [qml.expval(qml.PauliZ(wires=i)) for i in range(n_qubits)]


In [ ]:
def draw_circuit(qnn, n_qubits, weights):
    sampl_input = np.random.uniform(low=0, high=2*np.pi, size=(n_qubits,))
    weights = np.random.uniform(low=0, high=2*np.pi, size=weight_shapes["weights"])
    print("Sample of Ansatz circuit:\n")
    print(qml.draw(qnn)(sampl_input, weights))
    print("\n#########\n")

In [ ]:
draw_circuit(strong_entangling, n_qubits, weight_shapes)

## Integrating Pennylane Circuit into Keras

We integrate the PennyLane quantum node (`qnode`) into a TensorFlow Keras `Sequential` model using `qml.qnn.KerasLayer`. This allows the quantum circuit to be trained using standard classical backpropagation optimizers like Adam.


In [ ]:
def create_quantum_model(n_qubits, weights, prev):
    input_layer = tf.keras.layers.Input(shape=(n_qubits,))

    dev = qml.device("default.qubit", wires=n_qubits)
    qnode = qml.QNode(strong_entangling, dev, interface="tf", diff_method="backprop")

    # Convert the PennyLane QNode into a Keras Layer for hybrid training
    qlayer = qml.qnn.KerasLayer(qnode, weights, output_dim=n_qubits)

    activation=tf.keras.layers.Activation(tf.keras.activations.relu)
    output_layer = tf.keras.layers.Dense(len(prev), LeakyReLU(alpha=0.01))

    # Compile the hybrid model using the Adam optimizer and Mean Squared Error loss
    opt = tf.keras.optimizers.Adam(learning_rate = 0.001)

    # Everything that creates variables should be under the strategy scope.
    # In general this is only model construction & `compile()`.
    model = tf.keras.models.Sequential([
        input_layer
        , qlayer
        , activation
        , output_layer])

    model.compile(loss=['mse'], optimizer=opt, metrics=['mae'])

    return model

In [ ]:
model = create_quantum_model(n_qubits, weight_shapes, prevision_window)
input_shape = (n_qubits,)
model.build(input_shape)
model.summary()

## Model Training and Checkpointing

The hybrid model is trained using `EarlyStopping` to prevent overfitting and `ModelCheckpoint` to save the optimal weights derived from the validation set.


In [ ]:
# Halt training if the validation loss does not improve for 6 consecutive epochs
es=EarlyStopping(monitor='val_loss', min_delta=0, patience=6, verbose=0, mode='auto', baseline=None, restore_best_weights=True)

checkpoint_path = os.path.join('checkpoint', SUBDIR, f"cp-{ansatz}-lookback-{lookback}-depth-{n_layers}-"+"{epoch:04d}.ckpt")
os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)

# Create a callback that saves the model's weights
n_batches = len(Xs_train) / batch_size
n_batches = int(np.ceil(n_batches))
cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_path
                                                    , save_weights_only=True
                                                    , verbose=0
                                                    , save_freq=n_batches)
history_model = model.fit(Xs_train, y_train
                    , epochs=100
                    , batch_size=batch_size
                    , validation_data=(Xs_val, y_val)
                    , callbacks=[es]
                    , verbose=1)

### Plotting Loss

In [ ]:
plot_history(n_layers, history_model, ansatz, lookback, batch_size)

## Prediction

Prediction intervals were estimated using quantile regression applied to the errors of the validation datasets. A confidence index $q = 0.95$ was adopted to establish the upper and lower limits.


In [ ]:
path_an = os.path.abspath(os.path.join(os.getcwd(), 'analysis', SUBDIR))
os.makedirs(path_an, exist_ok=True)

print("\nTesting Normal Mean Limit\n")

In [ ]:
y_pred = model.predict(Xs_test,verbose=0)
mean_predictions, mean_error_normal, mean_error_left_normal, mean_error_right_normal = get_mean_left_right_error_interval(
    model
    , Xs_val
    , y_val
    , y_test
    , y_pred
)

In [ ]:
error_interval = [[str(i+1)+" hours"
                              , mean_predictions[0][i]
                              , mean_error_normal[0][i]
                              , mean_error_left_normal[0][i]
                              , mean_error_right_normal[0][i]] for i in range(len(prevision_window))]
interval_df = pd.DataFrame(error_interval, columns=['Horizon', 'Mean_Prediction', 'Mean_Error_Norm', 'Mean_Error_Left', 'Mean_Error_Right'])
interval_df.set_index("Horizon")

In [ ]:
filename = f"pred_interval-{ansatz}-lookback-{lookback}-batch-{batch_size}-depth-{n_layers}.csv"
print(f"Saving Metrics in {os.path.join(path_an,filename)}")
interval_df.to_csv(os.path.join(path_an,filename))

In [ ]:
plot_prediction_versus_observed(n_layers, y_test, y_pred, mean_error_normal, ansatz, lookback, batch_size, prevision_window)

## Statistical Analysis

The evaluation of our proposed Quantum Machine Learning (QML) method involves a rigorous quantitative analysis utilizing a suite of widely recognized statistical metrics tailored for time series data. We calculate NMSE, NRMSE, Pearson R correlation, R squared, and the Factor of 2 (FAC2) to evaluate predictive fidelity.

In [ ]:
all_analysis = quantitative_analysis(y_test, y_pred)
all_analysis

In [ ]:
filename = f"metrics-{ansatz}-lookback-{lookback}-batch-{batch_size}-depth-{n_layers}.csv"
print(f"Saving Metrics in {os.path.join(path_an,filename)}")
all_analysis.to_csv(os.path.join(path_an,filename))
print("\n#########\n")